# 04 - Embeddings & Semantic Search

**Learning objective:** Embed drug label chunks with a serverless embedding model and run semantic search over them.

We will:
1. Point an embeddings client at Scaleway Generative APIs (BGE Multilingual Gemma2, serverless)
2. Batch-embed all chunks
3. Run canonical similarity queries

### 🤖 Initialize the Embeddings Client

Point the `EmbeddingsClient` at **Scaleway Generative APIs** (`api.scaleway.ai`) using the **BGE Multilingual Gemma2** model. No infrastructure to provision - it is pay-per-token.

The client returns 3584-dim vectors natively; `EmbeddingsClient` truncates to **768 dimensions** to match the `chunks.embedding` column and the showcase 3 convention. A short round-trip confirms the endpoint is live.

In [ ]:
import os
import sys
import subprocess
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
sys.path.insert(0, "..")

from src.embeddings import EmbeddingsClient

embedding_openai = OpenAI(
    base_url="https://api.scaleway.ai/v1",
    api_key=os.environ["SCW_SECRET_KEY"],
)

embeddings = EmbeddingsClient(
    client=embedding_openai,
    model="bge-multilingual-gemma2",
    dimensions=768,  # client-side truncation from native 3584
)

test_emb = embeddings.embed("test")
print(f"Embedding dimension: {len(test_emb)}")

### 🐘 Reconnect to PostgreSQL

Reconnects to the Scaleway Managed PostgreSQL database and registers the `pgvector` extension,
enabling the database to store and compare embedding vectors.

In [ ]:
# Reconnect to database
import psycopg
from pgvector.psycopg import register_vector

pg_dir = "../iac_snippets/postgres"


def get_pg_output(name):
    r = subprocess.run(["tofu", "output", "-raw", name], cwd=pg_dir, capture_output=True, text=True)
    return r.stdout.strip()


conn = psycopg.connect(
    host=get_pg_output("host"),
    port=int(get_pg_output("port")),
    dbname=get_pg_output("database"),
    user=get_pg_output("user"),
    password=get_pg_output("password"),
    sslmode="require",
)
register_vector(conn)
print("Connected to database.")

### 🔢 Embed All Chunks & Store in Database

This cell fetches all chunks that do not yet have an embedding, generates vectors, and writes them back to the `chunks` table.

**Batch Processing:**
The code processes data in **batches of 32 chunks** at a time. Instead of sending each chunk individually, grouping them into "batches" makes the process significantly faster by reducing the number of API requests. After each batch, the changes are committed to the database.

> 💡 Texts longer than 2000 characters are truncated before embedding to ensure they fit within the model's limits.

In [ ]:
# Batch-embed all chunks and update the database
with conn.cursor() as cur:
    cur.execute("SELECT id, text FROM chunks WHERE embedding IS NULL")
    rows = cur.fetchall()

print(f"Chunks to embed: {len(rows)}")

# Process in batches
batch_size = 32
for i in range(0, len(rows), batch_size):
    batch = rows[i : i + batch_size]
    texts = [row[1][:2000] for row in batch]  # Truncate very long texts
    embs = embeddings.embed_batch(texts)

    with conn.cursor() as cur:
        for (chunk_id, _), emb in zip(batch, embs):
            cur.execute(
                "UPDATE chunks SET embedding = %s WHERE id = %s",
                (emb, chunk_id),
            )
    conn.commit()

    if (i // batch_size) % 5 == 0:
        print(f"  Embedded {min(i + batch_size, len(rows))}/{len(rows)} chunks")

print("All chunks embedded!")

### 🔍 Similarity Search: Drug Interaction Query

Tests the vector search with a real-world question:
*"Can I take ibuprofen with BUSPIRONE HYDROCHLORIDE?"*

The query is embedded and matched against the closest chunk in the database.
The result shows which drug label section is most semantically relevant.

In [ ]:
# Canonical query 1: "Can I take ibuprofen with BUSPIRONE HYDROCHLORIDE?"
from src.rag import similarity_search

query = "Can I take ibuprofen with BUSPIRONE HYDROCHLORIDE?"
query_emb = embeddings.embed(query)
results = similarity_search(conn, query_emb, k=1)

print(f"Query: {query}")
print("===")
for r in results:
    print(f"  Drug: {r['drug_name']} | Section: {r['section_type']} | Distance: {r['distance']:.3f}")
    print()

In [ ]:
# Canonical query 2: "8 USE IN SPECIFIC POPULATIONS Renal impairment"

query = "8 USE IN SPECIFIC POPULATIONS Renal impairment"
query_emb = embeddings.embed(query)
results = similarity_search(conn, query_emb, k=1)

print(f"Query: {query}")
print("===")
for r in results:
    print(f"  Drug: {r['drug_name']} | Section: {r['section_type']} | Distance: {r['distance']:.3f}")
    print()

### ✅ You should now have

- ☑️ Connected an embeddings client to Scaleway Generative APIs (BGE Multilingual Gemma2)
- ☑️ Generated 768-dimensional vectors for all FDA label chunks
- ☑️ Stored embeddings in PostgreSQL with pgvector
- ☑️ Tested similarity search with real clinical queries


**Next:** `05_tools.ipynb`